# 01 — GPU runs (plan.md Stages 3–7, task T2)

One notebook for every GPU stage: set the parameters, *Save & Run All (Commit)*, download the output.

**Kaggle settings:** Accelerator **GPU T4 ×2** · Internet **on** · secret `cvpr` · input dataset `medprune-data` ·
to resume a block, also add the previous session's notebook output as an input and set `RESTORE_FROM`.

| Stage | EXP | MODEL | Role / account |
|---|---|---|---|
| 3 | E0 (set `LIMIT = 50` for the first pass) | lingshu7b; qwen3vl8b | A; B |
| 4 | E1 | lingshu7b | A |
| 5 | E3, then E4 | lingshu7b; qwen3vl8b | A; B |
| 6 | E5, E6 (`ACCOUNT` = "A" / "B"), E7 | both | A, B |
| 7 | E6 (`ACCOUNT = "C"`), E8 | lingshu7b; huatuo7b | C |
| 5–6 | E7, E10 (resolution), E11 (flip, bottom-k); E10r, E11r on A/B | both | C; A, B |
| 6 | E12 (layer profile, text readout; §A10 D1–D2) | lingshu7b; qwen3vl8b | A; B |
| 6 | E13 (SparseVLM, VisPruner, PruneSID, BTP; second slot, notebook `medprune-baselines-<model>`) | lingshu7b; qwen3vl8b | A; B |
| 6 | E14 (VisionZip, HiPrune, CDPruner, FEATHER, VScan, TransPrune; `KEEP_GOING = True`) | both | D; B |
| 6 | E15 (`rin_open`, `JUDGE = True`), E16 (SparseVLM raters ablation) | lingshu7b; both | C; A |

In [ ]:
EXP = "E0"            # E0, E1, E1g, E3, E4, E5, E6, E7, E8, E8r, E10, E10r, E11, E11r, E12, E13; several run in block order: "E6,E3"
MODEL = "lingshu7b"   # lingshu7b | qwen3vl8b | huatuo7b; several load in turn: "qwen3vl8b,lingshu7b"
ACCOUNT = None        # role filter "A" / "B" / "C" / "D" (needed for E6 and E14, which span roles)
MAX_HOURS = 5.5       # stop scheduling new runs after this; the 12-hour Kaggle limit stays far away
LIMIT = None          # questions per run, e.g. 5 for a smoke test, 50 for E0's first pass
RESTORE_FROM = None   # "/kaggle/input/<previous-run-notebook>", or "auto": the attached dataset holding
                      # medprune_restore.json (run files + static maps uploaded from the laptop)
RUN_CHECKS = EXP == "E0"
NEEDS_RIN = bool(set(EXP.split(",")) & {"E1", "E1g", "E3", "E5", "E6", "E8r", "E10r", "E11r", "E12", "E13",
                                     "E14", "E14o", "E15", "E15b", "E16", "E17", "E17b", "E18", "E19"})
KEEP_GOING = False    # a failing run is reported and the next one starts (new ports, E14)
JUDGE = False         # judge open answers in this session (E15: RadImageNet text must not leave the session)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("cvpr")
    print("HF token loaded")
except Exception as error:  # needed only for RadImageNet-VQA (gated)
    print("secret 'cvpr' not attached (Add-ons -> Secrets); RadImageNet steps will fail:", type(error).__name__)

In [ ]:
!rm -rf /tmp/cvpr && git clone --depth 1 https://github.com/pushkar1887/cvpr /tmp/cvpr
%cd /tmp/cvpr
!git log --oneline -1
!bash scripts/setup_env.sh

Locate the shared bundle (the mount path can differ), set the results folder, and restore the previous session's results if continuing a block.

In [ ]:
import glob
import json
import shutil
from pathlib import Path

DATA = Path(next(Path(p).parent for p in glob.glob("/kaggle/input/**/bundle_manifest.json", recursive=True)))
RIN = Path("/tmp/medprune-rin")
RES = Path("/kaggle/working/results")
RES.mkdir(parents=True, exist_ok=True)
print("bundle:", DATA)

if RESTORE_FROM == "auto":  # flat dataset: *.jsonl / *.meta.json -> runs/, *.npy -> static_maps/
    marker = sorted(glob.glob("/kaggle/input/**/medprune_restore.json", recursive=True))
    assert len(marker) == 1, f"expected one attached dataset with medprune_restore.json, found {marker}"
    flat = Path(marker[0]).parent
    expected = json.loads(Path(marker[0]).read_text())["files"]
    for sub, pattern in (("runs", "*.jsonl"), ("runs", "*.meta.json"), ("static_maps", "*.npy")):
        (RES / sub).mkdir(parents=True, exist_ok=True)
        for f in flat.glob(pattern):
            shutil.copy(f, RES / sub / f.name)
    copied = {f.name for f in (RES / "runs").iterdir()} | {f.name for f in (RES / "static_maps").iterdir()}
    assert set(expected) <= copied, f"missing from the restore dataset: {sorted(set(expected) - copied)[:5]}"
    print("restored", len(list((RES / "runs").glob("*.jsonl"))), "run files and",
          len(list((RES / "static_maps").glob("*.npy"))), "static maps from", flat)
elif RESTORE_FROM:
    previous = sorted(glob.glob(f"{RESTORE_FROM}/**/results/runs", recursive=True))
    assert previous, f"no results/runs under {RESTORE_FROM}"
    shutil.copytree(Path(previous[0]).parent, RES, dirs_exist_ok=True)
    print("restored", len(list((RES / "runs").glob("*.jsonl"))), "run files from", Path(previous[0]).parent)

RadImageNet-VQA is built inside the session under `/tmp` (license: never in the shared bundle or in `/kaggle/working`). Record the printed field values and counts in the Work log.

In [ ]:
EXPS = set(EXP.split(","))
DERIVED = Path("/tmp/medprune-derived")  # E10/E11 controls: resized or flipped copies, session-only
if NEEDS_RIN:
    !python -m medprune.data build --out {RIN} --work /tmp/medprune-work --only radimagenet
    !python -m medprune.data masks --bundle {RIN} --datasets rin_anat_mc,rin_abn_closed,rin_open --models {MODEL}
if "E10r" in EXPS:  # RadImageNet anatomy upsampled to 448² (stays under /tmp, like RadImageNet itself)
    !python -m medprune.data resize --source {RIN} --bundle {RIN} --datasets rin_anat_mc --size 448
    !python -m medprune.data masks --bundle {RIN} --datasets rin_anat_mc_s448 --models {MODEL}
if "E10" in EXPS:  # SLAKE and natural images downsampled to 224² (64 visual tokens, as RadImageNet)
    !python -m medprune.data resize --source {DATA} --bundle {DERIVED} --datasets slake_en,natural --size 224
    !python -m medprune.data masks --bundle {DERIVED} --datasets slake_en_s224,natural_s224 --models {MODEL}
if "E11" in EXPS:  # SLAKE upside down: token positions stay, content moves
    !python -m medprune.data vflip --source {DATA} --bundle {DERIVED} --datasets slake_en
    !python -m medprune.data masks --bundle {DERIVED} --datasets slake_en_vflip --models {MODEL}
if "E11r" in EXPS:
    !python -m medprune.data vflip --source {RIN} --bundle {RIN} --datasets rin_anat_mc
    !python -m medprune.data masks --bundle {RIN} --datasets rin_anat_mc_vflip --models {MODEL}
BUNDLES = f"--bundle {DATA}" + (f" --bundle {RIN}" if NEEDS_RIN else "")
BUNDLES += f" --bundle {DERIVED}" if EXPS & {"E10", "E11"} else ""
FILTER = f"--exp {EXP} --model {MODEL}" + (f" --account {ACCOUNT}" if ACCOUNT else "")

In [ ]:
!python -m medprune.run_eval --list {FILTER} {BUNDLES} --out {RES}

Stage 3 only: fp16 stability, inert logging, offline keep-sets (writes `e0_report__<model>__t4x2.json`).

In [ ]:
if RUN_CHECKS:
    !python -m medprune.checks --model {MODEL} {BUNDLES} --out {RES} --n 50

The runs. Stops cleanly before `MAX_HOURS`; rerun the notebook (with `RESTORE_FROM`) to continue.

In [ ]:
LIMIT_ARG = f"--limit {LIMIT}" if LIMIT else ""
KEEP_ARG = "--keep-going" if KEEP_GOING else ""
!python -m medprune.run_eval {FILTER} {BUNDLES} --out {RES} --max-hours {MAX_HOURS} {LIMIT_ARG} {KEEP_ARG}

In [ ]:
if JUDGE:  # open answers of this session's runs; items (with question text) stay in /tmp, only verdicts are saved
    !python -m medprune.judge export --results {RES} --out /tmp/judge_items.jsonl --verdicts /tmp/no_verdicts.jsonl
    !python -m medprune.judge run --items /tmp/judge_items.jsonl --out {RES}/judge/verdicts_session.jsonl

In [ ]:
import json
import pandas as pd

rows = [json.loads(p.read_text()) for p in sorted((RES / "runs").glob("*.meta.json"))]
summary = pd.DataFrame(rows)
if not summary.empty:
    summary = summary[summary["model"] == MODEL]
    display(summary[["run_id", "n_done", "n_manifest", "accuracy", "mean_s_per_q"]])
!python -m medprune.run_eval --list {FILTER} {BUNDLES} --out {RES}

In [ ]:
!rm -rf /tmp/medprune-rin /tmp/medprune-work && du -sh /kaggle/working/results

### Next
1. Download this version's output and merge `results/` into `D:\cvpr\results\` on the laptop.
2. If `--list` still shows pending runs: new session, add this output as an input, set `RESTORE_FROM`, run again.
3. Add a Work log entry (plan.md Part G) with the run IDs, accuracies and any problems.